# 04 — Retrieval Performance Evaluation

Recall@K, Precision@K, Hit@K, MRR and nDCG@K are computed on the raw retriever ordering **before** the common MIGRES filter. Final-iteration values refer to the last raw retrieval trace. The iterative cumulative metrics live in notebook 05. For method-vs-method comparisons on the same questions, paired bootstrap CIs and paired randomisation tests are reported.

In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml", ROOT / "configs" / "gemini.yaml"]


In [ ]:
RUN_DIRS_BY_DATASET={}  # {'hotpotqa': {'sar': Path(...), ...}, 'musique': {...}, 'natural_questions': {...}}
LIMIT=None


In [ ]:
from sar_migres.utils.config import load_config
from sar_migres.corpus.models import Corpus
from sar_migres.datasets.loader import load_phase_questions,dataset_statistics
from sar_migres.experiments.iterative import retrieval_metrics_from_run, paired_run_comparison
from sar_migres.evaluation.plotting import save_bar_plot, save_line_plot
PLOT_ROOT=ROOT/'outputs/experiments/04_retrieval_performance/plots'
base=load_config(*BASE_CONFIG)
if not RUN_DIRS_BY_DATASET:
    print('Preflight only. Set RUN_DIRS_BY_DATASET to completed run directories for each of the three datasets before loading data.')
else:
    corpus=Corpus.from_file(base.data.corpus_path,lazy=True,embed_template=base.embedding.embed_template,max_malformed_fraction=base.data.max_malformed_fraction,max_documents=base.data.max_documents)
    for DATASET in base.data.datasets:
        RUN_DIRS=RUN_DIRS_BY_DATASET.get(DATASET, {})
        items=load_phase_questions(DATASET,base.data.dataset_paths[DATASET],'final',limit=LIMIT,final_split=base.data.final_split,validation_fraction=base.data.validation_fraction,validation_seed=base.data.validation_seed)
        print('\n=== Dataset:',DATASET,'===')
        print('Dataset statistics:',dataset_statistics(items))
        for name,path in RUN_DIRS.items():
            print('\n###',name)
            metrics=retrieval_metrics_from_run(path,items,corpus)
            print(json.dumps(metrics,indent=2,allow_nan=True))
            metric_dir=PLOT_ROOT/DATASET
            metric_dir.mkdir(parents=True,exist_ok=True)
            (metric_dir/f'{name}_metrics.json').write_text(json.dumps(metrics,indent=2,allow_nan=True),encoding='utf-8')
            save_bar_plot(list(metrics['final_mean']),list(metrics['final_mean'].values()),'Score','Retrieval metrics: '+DATASET+' / '+name,metric_dir/f'{name}_metrics.png')
        if 'sar' in RUN_DIRS:
            from sar_migres.evaluation.aggregation import filter_interaction_metrics
            for name,path in RUN_DIRS.items():
                print(name,'filter interaction:',filter_interaction_metrics(path))
                if name!='sar':
                    print(name,'vs SAR:',json.dumps(paired_run_comparison(path,RUN_DIRS['sar'],items,corpus,metric='recall_at_k',seed=base.run.statistics_seed,n_resamples=base.run.bootstrap_resamples),indent=2,allow_nan=True))
